# Experimentos de Triagem PLN — CareSync

Notebook para testar o parser e o motor de regras com laudos de exemplo. Execute a partir da raiz do repositório (ou ajuste o `sys.path`).

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.parser import LabParser
from src.rules import MotorDeRegras

In [2]:
laudos = {
    'normal': 'HEMOGRAMA\nHemoglobina: 13,5 g/dL (VR: 12,0 a 15,5)\nPlaquetas: 250.000 /mm³ (VR: 150.000 a 450.000)',
    'atencao': 'PERFIL LIPÍDICO\nColesterol total: 232 mg/dL (VR: < 190)\nGlicose: 118 mg/dL',
    'critico': 'ELETRÓLITOS\nPotássio: 6,6 mEq/L\nSódio: 138 mEq/L',
}

linhas = []
for nome, texto in laudos.items():
    dados = LabParser.parse_texto(texto)
    tipo = LabParser.detectar_tipo_exame(texto, [d['marcador'].id for d in dados])
    resultado = MotorDeRegras.avaliar_triagem_completa(dados, tipo)
    for item in resultado.exames_extraidos:
        linhas.append({'laudo': nome, 'tipo': resultado.tipo_exame, 'geral': resultado.classificacao_geral.value, **item.model_dump(mode='json')})

pd.DataFrame(linhas)

,laudo,tipo,geral,id,parametro,valor,unidade,ref_min,ref_max,ref_fonte,status_item,direcao
0,normal,Hemograma,NORMAL,hemoglobina,Hemoglobina,13.5,g/dL,12.0,15.5,laudo,NORMAL,NaN
1,normal,Hemograma,NORMAL,plaquetas,Plaquetas,250000.0,/mm³,150000.0,450000.0,laudo,NORMAL,NaN
2,atencao,Perfil lipídico,ATENCAO,colesterol,Colesterol total,232.0,mg/dL,NaN,190.0,laudo,ATENCAO,alto
3,atencao,Perfil lipídico,ATENCAO,glicose,Glicose (jejum),118.0,mg/dL,70.0,99.0,base,ATENCAO,alto
4,critico,Eletrólitos,CRITICO,potassio,Potássio,6.6,mEq/L,3.5,5.1,base,CRITICO,alto
5,critico,Eletrólitos,CRITICO,sodio,Sódio,138.0,mEq/L,135.0,145.0,base,NORMAL,NaN
